# VLAAI — ROI-only frequency ablation (all bands combined)

**The third view, completing the pair with the "global" notebook.**
`kaggle_rerun_vlaai_freq_global.py` removes one band from all 64 channels
at once (band-only, no ROI breakdown, using a synthetic "AllChannels"
group). This notebook is the mirror image: remove **all frequency content
(0.5-30 Hz combined)** from one ROI's channels at a time -- ROI-only, no
band breakdown. One number per ROI (9 for VLAAI's real montage), not per
ROI x band (54 in the original) or per band (6 in the global notebook).

**Implementation note:** same trick as before -- `BANDS` is patched to a
single entry spanning the full 0.5-30 Hz range, but this time the REAL
montage ROI grouping is kept as-is (unlike the global notebook, which
overrode `rois` to one synthetic group). `run_subject_level_roi_frequency_stats`
doesn't care how many bands or ROIs are in play -- same function, same
reuse-everything-else approach.

**Kaggle setup requirements:** Internet enabled (git clone + pip install
only). No GPU needed, no Kaggle Secret, no dataset attachment.

Output: `/kaggle/working/xai_results_vlaai_freq_roi_only/frequency_roi_only_subject.csv`
(schema: `roi,mean_dp,cohens_d,wilcox_p,fdr_p,fdr_sig,...` -- `band` column
dropped since there's only one combined band by construction).

## 1. Clone repository + install dependencies

In [ ]:
import os
import subprocess
import sys

REPO_DIR = "/kaggle/working/AAD_XAI"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/manjithadulana98/AAD_XAI.git", REPO_DIR],
        check=True,
    )
else:
    print(f"Repository already cloned at {REPO_DIR}")

os.chdir(REPO_DIR)

try:
    import torch as _torch_preinstalled
    print(f"Pre-installed torch {_torch_preinstalled.__version__} found "
          f"(CUDA available: {_torch_preinstalled.cuda.is_available()}) -- "
          "keeping it; installing the rest of requirements.txt without touching torch.")
    with open("requirements.txt") as _f:
        _reqs_no_torch = [ln for ln in _f if ln.strip() and not ln.strip().lower().startswith("torch")]
    with open("/tmp/requirements_no_torch.txt", "w") as _f:
        _f.writelines(_reqs_no_torch)
    subprocess.run(["pip", "install", "-q", "-r", "/tmp/requirements_no_torch.txt"], check=True)
except ImportError:
    print("No pre-installed torch found -- installing requirements.txt as-is.")
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)

subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, "scripts"))

print("Setup done.")

## 2. Device check (CPU by design this time)

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE} (forced CPU -- matches the other rerun notebooks this week)")

## 3. Configuration + verify local data/model files

In [ ]:
from pathlib import Path
import time

RANDOM_SEED = 42
N_BOOT = 2000
FDR_ALPHA = 0.05

DATA_DIR = os.path.join(REPO_DIR, "data", "vlaai_dtu_npz")
H5_PATH = os.path.join(REPO_DIR, "models", "vlaai.h5")
MONTAGE_PATH = os.path.join(REPO_DIR, "config", "dtu_channel_montage.csv")

assert os.path.isdir(DATA_DIR), f"Missing data dir: {DATA_DIR}"
assert os.path.isfile(H5_PATH), f"Missing model: {H5_PATH}"
assert os.path.isfile(MONTAGE_PATH), f"Missing montage: {MONTAGE_PATH}"

OUT_DIR = Path("/kaggle/working/xai_results_vlaai_freq_roi_only")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output dir: {OUT_DIR}  (scoped rerun -- NOT xai_results/)")

import numpy as np
import random
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

## 4. Import `run_focused_xai` as a module and patch `BANDS` to one
   combined range

In [ ]:
from collections import OrderedDict
import run_focused_xai as rfx

rfx.BANDS = OrderedDict([("all_0.5_30hz", (0.5, 30.0))])
print(f"Patched BANDS: {dict(rfx.BANDS)}")
assert len(rfx.BANDS) == 1

## 5. Load the REAL montage (unlike the global notebook, ROIs are kept
   as-is here), data, and model

In [ ]:
montage = rfx.load_montage(MONTAGE_PATH)
print(f"Montage: {len(montage['rois'])} ROIs: {list(montage['rois'])}")

print("\nLoading dataset (all windows, matching the original --max-samples -1)...")
from aad_xai.data.vlaai_dataset import VLAAIDTUDataset

ds = VLAAIDTUDataset(data_dir=DATA_DIR, window_length=320, hop=64, subjects=None)
N = len(ds)
selected_indices = list(range(N))
selected_subject_ids = np.asarray([ds.subject_ids[i] for i in selected_indices])
n_subjects = len(set(selected_subject_ids.tolist()))
print(f"Total windows: {N}  across {n_subjects} subjects")

t_load_start = time.time()
eeg_all = torch.stack([ds[i][0] for i in selected_indices]).to(DEVICE)
att_all = torch.stack([ds[i][1] for i in selected_indices]).to(DEVICE)
unatt_all = torch.stack([ds[i][2] for i in selected_indices]).to(DEVICE)
print(f"Windows loaded in {time.time() - t_load_start:.1f}s. eeg_all shape: {tuple(eeg_all.shape)}")

print("\nLoading VLAAI model...")
from aad_xai.models import VLAAIPyTorch, AADDecisionEEGOnly

model = VLAAIPyTorch.from_h5(H5_PATH)
model.eval().to(DEVICE)

decision = AADDecisionEEGOnly(model)
decision.eval().to(DEVICE)

decision.set_envelopes(att_all[:3], unatt_all[:3])
with torch.no_grad():
    test_logits = decision(eeg_all[:3])
print(f"Smoke-test decision logits: {test_logits[0].detach().cpu().numpy()}")

decision.set_envelopes(att_all, unatt_all)
print("Model loaded and verified.")

## 6. Run the ROI-only ablation (1 band x 9 ROIs = 9 combos)

In [ ]:
t_start = time.time()
freq_stats = rfx.run_subject_level_roi_frequency_stats(
    decision, eeg_all, att_all, unatt_all,
    selected_subject_ids, montage,
    OUT_DIR, FDR_ALPHA, N_BOOT, RANDOM_SEED,
)
t_total = time.time() - t_start
print(f"\nTotal wall-clock time: {t_total:.1f}s ({t_total / 60:.1f} min)")

## 7. Verify + clean up output, write timing record

In [ ]:
import pandas as pd
import json

written_path = OUT_DIR / "subject_level_roi_frequency_stats.csv"
final_path = OUT_DIR / "frequency_roi_only_subject.csv"
assert written_path.exists(), f"Expected output not found: {written_path}"

df = pd.read_csv(written_path)
assert set(df["band"].unique()) == {"all_0.5_30hz"}, f"Expected only one band, got {df['band'].unique()}"
df = df.drop(columns=["band"])  # only one value by construction, not informative as a column
df.to_csv(final_path, index=False)
written_path.unlink()

roi_values = sorted(df["roi"].unique().tolist())
print(f"ROIs present ({len(roi_values)}): {roi_values}")
assert len(roi_values) == 9, f"Expected 9 ROIs, got {len(roi_values)}: {roi_values}"
n_sig = int(df["fdr_sig"].sum())
print(f"FDR-significant ROIs: {n_sig}/{len(df)}")
print(df.to_string(index=False))

with open(OUT_DIR / "rerun_timing.json", "w") as f:
    json.dump({
        "total_seconds": t_total,
        "total_minutes": t_total / 60,
        "n_windows": N,
        "n_subjects": n_subjects,
        "n_rois": len(montage["rois"]),
        "full_band_hz": [0.5, 30.0],
        "n_boot": N_BOOT,
        "fdr_alpha": FDR_ALPHA,
        "seed": RANDOM_SEED,
        "purpose": "ROI-only ablation (all frequency content combined) -- no band dimension",
        "montage_file": "config/dtu_channel_montage.csv",
        "device": "cpu",
    }, f, indent=2)

print(f"\nWritten to {final_path}")
print(f"Timing/config record: {OUT_DIR / 'rerun_timing.json'}")